# Exercise 1.1 — Exploration and Representation

**Marketing campaigns** · `data/creadores.csv`

> *Analyse the dataset and select which variables will be used to solve the problem. To do so, perform an Exploratory Data Analysis (EDA) and a Principal Component Analysis (PCA). Justify whether dimensionality reduction is applied and how the components are interpreted.*

**Contents**

1. Setup
2. Data and variable selection
3. Exploratory Data Analysis (EDA)
4. Standardisation
5. Principal Component Analysis (PCA)
6. Conclusions and hand-off

All reusable logic lives in `src/tp4/` (`data.py`, `linear/eda.py`, `linear/pca.py`); this notebook only calls it and shows the results.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from tp4.data import load_creators, standardise
from tp4.linear.eda import plot_distributions, plot_boxplots, plot_correlation_matrix, high_correlation_pairs
from tp4.linear.pca import fit_pca, loadings_df, scree_plot, plot_loadings, biplot, bootstrap_stability

plt.rcParams["figure.dpi"] = 120
DATA_PATH = "../data/creadores.csv"

## 2. Data and variable selection

`load_creators()` reads the CSV and removes two kinds of columns:

**Identifiers** — label a row but say nothing about the creator:
- `creator_id`, `name`

**Deterministic derivatives** — fully determined by columns we keep, so they add no new information and would double-count it in PCA:
- `price_ars` = `price_usd × 1500` → keep `price_usd`
- `organic_posts_pct` = `100 − sponsored_posts_pct` → keep `sponsored_posts_pct`
- `total_engagement_per_1000` = sum of the four engagement columns → keep the four

**Kept on purpose, although derived from other columns:**
- `estimated_argentina_reach` = `average_reach × argentina_audience_pct / 100`. Unlike the columns above it is a *product* of two varying columns, not a copy or a linear combination, so PCA and Kohonen cannot reconstruct it. It measures how many Argentine accounts a post actually reaches, which matters for clients targeting Argentina.
- All four `topic_*_pct` and all four `content_*_pct` columns (see section 3.3).

In [ ]:
raw = pd.read_csv(DATA_PATH)
creator_names = raw["name"].tolist()   # kept separately for the biplot labels

df = load_creators(DATA_PATH)

print(f"Raw data:      {raw.shape[0]} creators × {raw.shape[1]} columns")
print(f"After cleanup: {df.shape[0]} creators × {df.shape[1]} features")
print(f"Missing values: {df.isna().sum().sum()}   Duplicate rows: {raw.duplicated().sum()}")

In [ ]:
# Range and spread of every feature, in original units
df.describe().T[["mean", "std", "min", "max"]].round(2)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distributions

Look for skewed variables (e.g. audience size) and outliers before running PCA.

In [ ]:
plot_distributions(df)

### 3.2 Correlations

Strongly correlated variables carry overlapping information and will load on the same principal component.

In [ ]:
plot_correlation_matrix(df)

In [ ]:
# Pairs with |correlation| > 0.8
high_correlation_pairs(df, threshold=0.8)

### 3.3 Groups of percentages that always sum to 100

The four `topic_*_pct` columns and the four `content_*_pct` columns each describe how a creator's posts are split. If every row sums to 100, one column per group is determined by the other three — the same situation as `organic_posts_pct`.

**Decision:** we keep all four columns in both groups, so that every topic (one per client) and every content type stays directly interpretable in the loadings and the biplot. As a consequence, the last two principal components have zero variance.

In [ ]:
groups = {
    "topic":   [c for c in df.columns if c.startswith("topic_")],
    "content": [c for c in df.columns if c.startswith("content_")],
}
for name, cols in groups.items():
    row_sums = df[cols].sum(axis=1)
    print(f"{name:8s} {cols}")
    print(f"         row sums: min {row_sums.min():.2f}, max {row_sums.max():.2f}")

## 4. Standardisation

PCA is sensitive to scale: `followers` (up to ~775 000) would dominate `likes_per_1000` (up to ~55) only because of its units. Every feature is therefore z-scored to mean 0 and standard deviation 1.

In [ ]:
X_scaled, scaler = standardise(df)

# StandardScaler divides by the population std (ddof=0), so check with numpy's default
means, stds = X_scaled.mean(axis=0), X_scaled.std(axis=0)
print(f"means in [{means.min():.3f}, {means.max():.3f}],  stds in [{stds.min():.3f}, {stds.max():.3f}]")

### 4.1 Boxplots before and after standardisation

The same comparison as in the course slides (5 - PCA, slides 43–44). On the raw scale, the audience-size variables (up to ~775 000) squeeze every other variable into a flat line at zero. After standardisation all variables share one scale, so their spread, skewness and outliers can be compared.

In [ ]:
plot_boxplots(df, "Before standardisation (original units)", "Value in original units (followers, USD, %, ...)")

In [ ]:
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)
plot_boxplots(X_scaled_df, "After standardisation (z-scores)", "Standard deviations from the mean")

## 5. Principal Component Analysis (PCA)

In [ ]:
pca = fit_pca(X_scaled)
loadings = loadings_df(pca, df.columns.tolist())

explained = pd.DataFrame(
    {"explained_variance": pca.explained_variance_ratio_},
    index=loadings.columns,
)
explained["cumulative"] = explained["explained_variance"].cumsum()
explained.head(10).round(3)

### 5.1 Scree plot

How many components are needed? Look for the elbow and for where the cumulative curve passes 70–80 %.

In [ ]:
scree_plot(pca)

### 5.2 Loadings of PC1 and PC2

A loading is the weight of an original variable in a component. Large positive → high values of that variable push the score up; large negative → push it down.

In [ ]:
loadings[["PC1", "PC2"]].sort_values("PC1").round(3)

In [ ]:
plot_loadings(loadings, ["PC1", "PC2"])

### 5.3 Sensitivity check: PCA without the Argentine variables

`estimated_argentina_reach` and `argentina_audience_pct` were kept on purpose (section 2). To see what they contribute, we rerun the PCA without them and compare PC1 and PC2.

In [ ]:
ARGENTINE_COLS = ["estimated_argentina_reach", "argentina_audience_pct"]
df_no_arg = df.drop(columns=ARGENTINE_COLS)

X_no_arg, _ = standardise(df_no_arg)
pca_no_arg = fit_pca(X_no_arg)
loadings_no_arg = loadings_df(pca_no_arg, df_no_arg.columns.tolist())


def pole_summary(load: pd.DataFrame, pc: str, k: int = 3) -> str:
    # The k most negative and k most positive variables of one component
    s = load[pc].sort_values()
    return f"{', '.join(s.index[:k])}  ↔  {', '.join(s.index[-k:][::-1])}"


comparison = pd.DataFrame({
    "with Argentine variables (28)": [
        f"{pca.explained_variance_ratio_[0]:.1%}", pole_summary(loadings, "PC1"),
        f"{pca.explained_variance_ratio_[1]:.1%}", pole_summary(loadings, "PC2"),
    ],
    "without them (26)": [
        f"{pca_no_arg.explained_variance_ratio_[0]:.1%}", pole_summary(loadings_no_arg, "PC1"),
        f"{pca_no_arg.explained_variance_ratio_[1]:.1%}", pole_summary(loadings_no_arg, "PC2"),
    ],
}, index=["PC1 variance", "PC1 poles", "PC2 variance", "PC2 poles"])

with pd.option_context("display.max_colwidth", None):
    display(comparison)

PC1 hardly changes: the size ↔ depth axis does not depend on the Argentine variables. PC2 does change: without them it describes content type instead of Argentine audience and topic. The Argentine variables therefore add information that no other variable carries, which supports keeping them.

### 5.4 Biplot (PC1 vs. PC2)

Points are creators; arrows are the original variables. Creators lie in the direction of the variables they score high on. This plot is the reference for Exercise 1.2 (Kohonen).

In [ ]:
biplot(pca, X_scaled, df.columns.tolist(), labels=creator_names)

### 5.5 How stable are the components?

With only 50 creators, components whose eigenvalues lie close together can swap or mix. We resample the creators 300 times (bootstrap), refit the PCA and measure how similar each new component is to the original one (|cos|: 1 = same direction, 0 = unrelated).

In [ ]:
stability = bootstrap_stability(df, n_components=8, n_boot=300, seed=0)
stability.round(2)

PC1 is practically identical in every resample and PC2 is mostly the same. From PC3 on, the components change from sample to sample: their eigenvalues (1.9, 1.6, 1.5, 1.2) are too close for the order and the mix of variables to be well determined with 50 creators. They should not be interpreted one by one.

### 5.6 Which groups of variables can be compressed?

PCA can only compress variables that overlap. Running it on three groups separately shows where a reduction would lose information and where it would not.

In [ ]:
variable_groups = {
    "size":  ["followers", "average_reach", "average_impressions", "average_video_views",
              "estimated_argentina_reach", "price_usd"],
    "depth": ["average_caption_words", "saves_per_1000", "comments_per_1000",
              "profile_visits_per_1000", "average_watch_time_seconds"],
    "topic": ["topic_football_pct", "topic_makeup_pct", "topic_dogs_pct", "topic_art_pct"],
}
group_rows = {}
for group, cols in variable_groups.items():
    group_pca = fit_pca(standardise(df[cols])[0])
    group_rows[group] = {f"PC{i+1}": f"{v:.0%}" for i, v in enumerate(group_pca.explained_variance_ratio_[:3])}
pd.DataFrame(group_rows).T

The six size variables collapse into one component (91 %), and the depth variables almost do (75 %). The four topics give three components of about one third each: they are independent, so PCA cannot compress them without dropping a topic.

## 6. Conclusions

### 6.1 Variable selection
- **28 of 33 columns are used.** Removed: the identifiers `creator_id` and `name`, and three exact functions of kept columns (`price_ars`, `organic_posts_pct`, `total_engagement_per_1000`).
- **Kept on purpose:** `estimated_argentina_reach` (a product of two varying columns, not reconstructable by PCA or Kohonen, and relevant for clients in Argentina) and all four `topic_*` and `content_*` columns (each topic maps to a client). The two groups summing to 100 cause two zero-variance components.
- **Clean data:** no missing values, no duplicate rows.

### 6.2 What the EDA shows
- **Very different scales** (followers 9 604 – 775 589, price 75 – 6 335 USD) → standardisation is required (section 4.1).
- **Two blocks of strongly correlated variables:** size (followers, reach, impressions, video views, Argentine reach, price; r = 0.70 – 1.00) and format (video, short-form, trend, likes; r = 0.87 – 0.97). Depth variables (caption length, saves, comments, profile visits, watch time) are negatively correlated with size. Many strong correlations mean PCA is meaningful.
- **Creators differ by main topic:** dog creators are small and cheap, write long captions and tutorials and have the most Argentine audience; makeup creators are the largest, with short entertaining videos.

### 6.3 PCA results and interpretation
- **PC1 (49.9 %) = size and virality ↔ depth and engagement.** Large accounts with high reach and trend/short-form posts on one side; long captions, saves, comments and profile visits on the other. It correlates −0.91 with log(followers).
- **PC2 (9.4 %) = Argentine audience and dogs ↔ art and showcase content.** Without the two Argentine variables PC1 is unchanged but PC2 becomes a content-type axis (section 5.3), so these variables add information nothing else carries.
- **Number of components:** the elbow is right after PC1; 4 components explain 70 %, 6 explain 80 %, and 6 have an eigenvalue above 1 (Kaiser).
- **PC1 and PC2 are stable; PC3 onwards is not** (section 5.5). Together PC2 – PC5 hold the differences between topics, but how that information is split across them changes with the sample.

### 6.4 Dimensionality reduction: decision

| For reducing | Against reducing |
|---|---|
| The six size variables count six times in a distance-based method; PCA merges them | The task asks Kohonen to group creators *"en términos de todas sus variables"* |
| Small components may be noise | Topic information, what separates the four clients, sits in PC2 – PC5; 2 components (59 %) would lose most of it |
| Fewer inputs | PC3 onwards is unstable, so a map built on components cannot be interpreted per component |
| | Topics cannot be compressed (section 5.6) |
| | 50 × 28 is small: no computational reason to reduce |

**Decision: no dimensionality reduction before Kohonen.** The 28 standardised variables are the input; PCA is used to understand the data and as the reference for exercises 1.2 and 1.3. The cost is that the size block weighs heavily in distances, which must be kept in mind when interpreting the map.

### 6.5 Hand-off to the next exercises

#### Input data shared by all exercises

All exercises (1.2 Kohonen, 1.3 Oja, 1.4 decisions) must use the same variables and the same scaling. Load them with:

```python
from tp4.data import load_creators, standardise

df = load_creators("data/creadores.csv")     # 50 creators × 28 variables
X_scaled, scaler = standardise(df)           # z-scores: mean 0, std 1 per column
```

`load_creators()` in `src/tp4/data.py` is the **single source of truth** for the variable selection. Do not drop or add columns elsewhere: change `_ID_COLS` / `_REDUNDANT_COLS` there, so every exercise follows automatically, and update this list.

**The 28 variables used**

| Group | Variables |
|---|---|
| Size and reach | `followers`, `average_reach`, `average_impressions`, `average_video_views`, `estimated_argentina_reach`, `non_follower_reach_pct` |
| Commercial | `price_usd`, `sponsored_posts_pct`, `monthly_posts` |
| Audience | `argentina_audience_pct` |
| Engagement (per 1000 reached) | `likes_per_1000`, `comments_per_1000`, `saves_per_1000`, `shares_per_1000`, `profile_visits_per_1000` |
| Topic (sum to 100) | `topic_football_pct`, `topic_makeup_pct`, `topic_dogs_pct`, `topic_art_pct` |
| Content type (sum to 100) | `content_tutorial_pct`, `content_review_pct`, `content_entertainment_pct`, `content_showcase_pct` |
| Format and style | `video_posts_pct`, `short_form_posts_pct`, `trend_posts_pct`, `average_caption_words`, `average_watch_time_seconds` |

**Not used:** `creator_id`, `name` (identifiers; keep `name` aside to label results), `price_ars`, `organic_posts_pct`, `total_engagement_per_1000` (exact functions of kept columns).

**Column order matters.** Kohonen neuron weights and Oja weights come out in the order of `df.columns`, so weight *i* belongs to column *i*. The table below gives the index of every variable.

In [ ]:
# Index of every input variable: weight i of a Kohonen neuron or of Oja's rule belongs to column i
pd.DataFrame({"variable": df.columns}).rename_axis("index")

**1.2 Kohonen**
- Input: `X_scaled` from above (50 × 28). Use exactly this, so both exercises use the same variables and scaling.
- Compare the map with the biplot in section 5.4 (the task asks whether the regions match PC1 and PC2). Expected profiles: large and viral vs. small and in-depth (PC1), Argentine audience and dogs vs. art and showcase (PC2), and the topic profiles in section 3.
- Distances are Euclidean, so the six size variables count about six times. If the map is organised almost only by size, that is the reason.

**1.3 Oja's rule**
- Same input as Kohonen. Oja's rule converges to the first eigenvector of the correlation matrix, so compare its weights with `pca.components_[0]` and the index with `pca.transform(X_scaled)[:, 0]`.
- The sign may come out opposite to ours (slide 46, *"autovector opuesto"*); compare the absolute cosine, or flip the sign.
- Interpretation: the index is mostly size versus depth and says nothing about topic. Choosing the creator with the highest value for every campaign is close to choosing by followers, the strategy the agency already uses.

**1.4 Campaign decisions**
- The follower-only strategy is roughly PC1; what separates the clients lies outside it.
- The topic profiles (section 3) and Argentine reach are the most direct evidence for matching creators to clients.
- Monthly updates: the standardisation and PCA must be refitted; means, loadings and possibly the sign of a component change. New variables require repeating the EDA (redundancy and correlations).